# Tutorial / A Minimal Demo

Here, we show how to apply GC and spike compression to a Conv1d SNN (SCNN) for Sequential CIFAR-10 direct training.

Before reading this tutorial, please get familiar with basic concepts like [containers in SpikingJelly](https://spikingjelly.readthedocs.io/zh-cn/latest/activation_based_en/container.html).

**IMPORTANT: we run this notebook on our PC, rather than on the server where we conduct the main experiment. Hence, the memory and time consumptions displayed in this notebook are slightly different from the results reported in our manuscript!**

## The Original SNN

This is the definition of the original SCNN, adopted from PSN (Fang et al., 2023).

In [1]:
import sys

sys.path.append("./src")

import torch
import torch.nn as nn
import torch.nn.functional as F
from spikingjelly.activation_based import layer, functional

from modules.neuron import get_neuron # see src/modules/neuron.py for details

CHANNELS = 128
DEVICE = "cuda:0"
EPOCHS = 300
LR = 0.1
MOMENTUM = 0.9


class SequentialCIFARNet(nn.Module):

    def __init__(
        self, channels: int, neuron_type: str, num_classes=100, **kwargs
    ):
        """A Conv1d-based network for Sequential CIFAR-10/100 classification.

        Args:
            channels (int)
            neuron_type (str)
            num_classes (int, optional): Defaults to 100.
            **kwargs: Additional arguments for `get_neuron(...)`. See 
                `src/models/neuron.py` for details.
        """
        super().__init__()

        conv = []
        for _ in range(2):
            for _ in range(3):
                if len(conv) == 0:
                    in_channels = 3
                else:
                    in_channels = channels

                conv_block = nn.Sequential(
                    layer.Conv1d(
                        in_channels,
                        channels,
                        kernel_size=3,
                        padding=1,
                        bias=True,
                        step_mode="m"
                    ), # multi-step layer. See SpikingJelly tutorials for details.
                    layer.BatchNorm1d(channels, step_mode="m"),
                    get_neuron(neuron_type, **kwargs),
                )
                conv.append(conv_block)
            conv.append(layer.AvgPool1d(2, 2, step_mode="m"))

        self.conv = nn.Sequential(*conv)

        self.fc = nn.Sequential(
            layer.Linear(channels * 8, channels * 8 // 4, step_mode="m"),
            get_neuron(neuron_type, **kwargs),
        )

        self.decode = nn.Linear(channels * 8 // 4, num_classes)

    def forward(self, x):
        x = x.permute(3, 0, 1, 2)
        # x.shape = [T, N, Cin, L]
        y = self.conv(x)
        y = y.flatten(start_dim=-2)  # [T, N, C*L]
        y = self.fc(y)  # [T, N, C']
        y = y.mean(dim=0)  # [N, C']
        y = self.decode(y)
        return y

sjlif_net = SequentialCIFARNet(
    channels=CHANNELS, 
    neuron_type="SJLIF",  # defined in src/modules/neuron.py; indeed the same as the LIFNode in SpikingJelly
    num_classes=10, 
    decay_lambda=0.5
).to(DEVICE)

print(sjlif_net)

TORCH_VERSION=2.1.2+cu118, DISABLE_COMPILE should be False
DISABLE_COMPILE is manually set to True. 
DEFAULT_BACKEND is manually set to inductor. 
Triton kernel with float8e4nv (float8_e4m3fn) is not supported on devices with compute capability < 8.9. Your devices's capability is: (8, 6).
Use Triton kernels for BitSpikeCompressor.
Using Triton kernels for HandWrittenLIF.
Using torch kernels for HQLIF.
torch.amp is not available. Use torch.cuda.amp instead.
DEFAULT_HQ_DTYPE: torch.float8_e4m3fn
Using torch backend for spikingjelly by default.
SequentialCIFARNet(
  (conv): Sequential(
    (0): Sequential(
      (0): Conv1d(3, 128, kernel_size=(3,), stride=(1,), padding=(1,), step_mode=m)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=m)
      (2): SJLIF(
        v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=torch, tau=2.0
        (surrogate_function): ATan(alpha=2.0, spiking=True)
      )
    )
    (1): Sequen

Let's train the SNN and inspect its peak memory usage.

* You don't have to fully understand what `SCIFARDataModule` is. Just keep in mind that `train_loader` and `val_loader` are the training and validation data loaders! 

In [2]:
sys.path.append("./src/scifar")

from tqdm import tqdm

from data_module import SCIFARDataModule
from utils import AverageMeter, accuracy

# get data loaders
lightning_data_module = SCIFARDataModule(data_dir="../datasets/CIFAR10", num_classes=10, batch_size=128, num_workers=4)
lightning_data_module.setup("fit")
train_loader = lightning_data_module.train_dataloader()
val_loader = lightning_data_module.val_dataloader()

def train_step(
    net, train_data_loader, optimizer, lr_scheduler, device, current_epoch
):
    net.train()
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    with tqdm(
        train_data_loader,
        desc=f"Epoch {current_epoch}",
        leave=False,
        unit="batch"
    ) as pbar:
        for img, label in pbar:
            img, label = img.float().to(device), label.to(device)

            y = net(img)
            batch_loss = F.cross_entropy(y, label)

            batch_loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            prec1, prec5 = accuracy(y.data, label.argmax(1).data, topk=(1, 5))
            losses.update(batch_loss.item(), label.size(0))
            top1.update(prec1.item(), label.size(0))
            top5.update(prec5.item(), label.size(0))

            pbar.set_postfix({
                "loss": losses.avg,
                "top1_acc": top1.avg,
                "top5_acc": top5.avg,
            })

    if lr_scheduler is not None:
        lr_scheduler.step()

    return {
        "loss": losses.avg,
        "top1_acc": top1.avg,
        "top5_acc": top5.avg,
    }


def val_step(net, test_data_loader, device):
    net.eval()
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    with torch.no_grad():
        for img, label in test_data_loader:
            img, label = img.float().to(device), label.to(device)

            y = net(img)
            batch_loss = F.cross_entropy(y, label)

            # measure accuracy and record loss
            functional.reset_net(net)
            prec1, prec5 = accuracy(y.data, label.data, topk=(1, 5))
            losses.update(batch_loss.item(), label.size(0))
            top1.update(prec1.item(), label.size(0))
            top5.update(prec5.item(), label.size(0))

    return {
        "loss": losses.avg,
        "top1_acc": top1.avg,
        "top5_acc": top5.avg,
    }

optimizer = torch.optim.SGD(
    sjlif_net.parameters(),
    lr=LR,
    momentum=MOMENTUM,
)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)


# reset peak memory recorder
torch.cuda.reset_peak_memory_stats(DEVICE)

for epoch in range(1): # only train 1 epoch
        train_results = train_step(
            sjlif_net,
            train_loader,
            optimizer,
            lr_scheduler,
            DEVICE,
            epoch,
        )
        val_results = val_step(
            sjlif_net,
            val_loader,
            DEVICE,
        )

        mem_stats = torch.cuda.memory_stats(DEVICE)
        peak_allocated = mem_stats["allocated_bytes.all.peak"] / (1024**2)
        peak_reserved = mem_stats["reserved_bytes.all.peak"] / (1024**2)

        print(
            f"Epoch {epoch + 1}: "
            f"train_loss={train_results['loss']}, "
            f"train_top1_acc={train_results['top1_acc']}, "
            f"val_loss={val_results['loss']}, "
            f"val_top1_acc={val_results['top1_acc']},\n\t"
            f"peak_allocated={peak_allocated} MB, "
            f"peak_reserved={peak_reserved} MB"
        )

Files already downloaded and verified
Files already downloaded and verified


Epoch 1: train_loss=2.0764770434452937, train_top1_acc=24.87580128205128, val_loss=1.6215279363632202, val_top1_acc=40.74,
	peak_allocated=1281.46142578125 MB, peak_reserved=1846.0 MB


## Apply Gradient Checkpointing and Spike Compression

Use the interface we provided in `src/modules/blocks/checkpointing.py` to wrap SNN blocks into GC segments.
* Alternatively, you can directly use the `BaseCheckpointingBlock` classes implemented in `src/modules/blocks`.

`SNNCheckpointingBlockFunction` is a customized `torch.autograd.Function` whose BP is defined to simulate gradient checkpointing.
```python
SNNCheckpointingBlockFunction.apply(func, x_compressor, x_seq, *args)
```
* `func`: the forward function of the SNN block / GC segment. `func` should have the signature `func(x_seq, *args)`
* `x_compressor`: spike compressor; see `src/modules/compress/spike_compressor.py`
* `x_seq`: the input spike sequence
* `args`: other arguments passed to `func`

`BaseCheckpointingBlock` is the base class for all checkpointing segments. To inherit from `BaseCheckpointingBlock`, you need to implement the following methods:
* `__init__(...)`: pass a series of `torch.nn.Modules` and the spike compressor to the constructor. The modules are the components of the GC segment.
* `conventional_forward(x_seq, *args)`: define the forward logic. It is a static method. Users should **pass weights or arguments to this function, rather than the component modules**. **Use the PyTorch functional APIs to implement the forward logic** inside this function. The last argument `conventional_forward` should be `in_backward` to distinguish the forward behavior in FP phase from the behavior in BP phase (GC calls `forward` the second time during BP phase); this argument will be automatically passes by `SNNCheckpointingBlockFunction`.
* `forward(self, x_seq)`: call `SNNCheckpointingBlockFunction.apply`. `conventional_forward` should act as the `SNNCheckpointingBlockFunction.apply`'s first argument.


In [3]:
from modules.blocks.checkpointing import BaseCheckpointingBlock
from modules.blocks.checkpointing import SNNCheckpointingBlockFunction
from modules.kernels import * # accelerated SNN kernels

class Conv1dBNLIF(BaseCheckpointingBlock):
    """A GC segment containing:
    * Conv1d
    * BatchNorm1d
    * LIF (SJLIF or HandWrittenLIF)
    """

    def __init__(
        self,
        proj: nn.Conv1d,
        bn: nn.BatchNorm1d,
        neuron: nn.Module,
        spike_compressor
    ):
        super().__init__()
        self.proj = proj
        self.bn = bn
        self.neuron = neuron
        self.spike_compressor = spike_compressor

    @staticmethod
    def conventional_forward(
        x_seq,
        weight,
        bias,
        stride,
        padding,
        dilation,
        groups,
        bn_weight,
        bn_bias,
        bn_running_mean,
        bn_running_var,
        training,
        neuron,
        in_backward=False
    ):
        x_seq = conv1d_bn_forward( # a SNN kernel defined in src/modules/kernels
            x_seq,
            weight,
            bias,
            stride,
            padding,
            dilation,
            groups,
            bn_weight,
            bn_bias,
            bn_running_mean,
            bn_running_var,
            training,
            momentum=0.1 if in_backward else 0. # Update BN statistics only in BP phase!!!
        )
        return neuron(x_seq)

    def forward(self, x_seq: torch.Tensor):
        return SNNCheckpointingBlockFunction.apply( 
            self.conventional_forward,
            self.spike_compressor,
            x_seq,
            self.proj.weight,
            self.proj.bias,
            self.proj.stride,
            self.proj.padding,
            self.proj.dilation,
            self.proj.groups,
            self.bn.weight,
            self.bn.bias,
            self.bn.running_mean,
            self.bn.running_var,
            self.bn.training,
            self.neuron,
        ) 
        # SNNCheckpointingBlockFunction.apply acts as a wrapper
        # It will call self.conventional_forward() with arguments from x_seq
        # to self.neuron in a gradient checkpointing style. Also, input spike
        # will be compressed by self.spike_compressor before saving for backward.


class AvgPool1dConv1dBNLIF(BaseCheckpointingBlock):
    """A GC segment containing:
    * AvgPool1d
    * Conv1d
    * BatchNorm1d
    * LIF (SJLIF or HandWrittenLIF)
    """

    def __init__(
        self,
        pool: nn.AvgPool1d,
        proj: nn.Conv1d,
        bn: nn.BatchNorm1d,
        neuron: nn.Module,
        spike_compressor
    ):
        super().__init__()
        self.pool = pool
        self.proj = proj
        self.bn = bn
        self.neuron = neuron
        self.spike_compressor = spike_compressor

    @staticmethod
    def conventional_forward(
        x_seq,
        pool_kernel_size,
        pool_stride,
        pool_padding,
        weight,
        bias,
        stride,
        padding,
        dilation,
        groups,
        bn_weight,
        bn_bias,
        bn_running_mean,
        bn_running_var,
        training,
        neuron,
        in_backward=False
    ):
        y_seq = avgpool1d_conv1d_bn_forward( # a SNN kernel defined in src/modules/kernels
            x_seq,
            pool_kernel_size,
            pool_stride,
            pool_padding,
            weight,
            bias,
            stride,
            padding,
            dilation,
            groups,
            bn_weight,
            bn_bias,
            bn_running_mean,
            bn_running_var,
            training,
            momentum=0.1 if in_backward else 0. # Update BN statistics only in BP phase!!!
        )
        return neuron(y_seq)

    def forward(self, x_seq: torch.Tensor):
        return SNNCheckpointingBlockFunction.apply(
            self.conventional_forward,
            self.spike_compressor,
            x_seq,
            self.pool.kernel_size[0],
            self.pool.stride[0],
            self.pool.padding[0],
            self.proj.weight,
            self.proj.bias,
            self.proj.stride,
            self.proj.padding,
            self.proj.dilation,
            self.proj.groups,
            self.bn.weight,
            self.bn.bias,
            self.bn.running_mean,
            self.bn.running_var,
            self.bn.training,
            self.neuron,
        )


class AvgPool1dFlattenLinearLIF(BaseCheckpointingBlock):
    """A GC segment containing:
    * AvgPool1d
    * Flatten
    * Linear
    * LIF (SJLIF or HandWrittenLIF)
    """

    def __init__(
        self,
        pool: nn.AvgPool1d,
        proj: nn.Linear,
        neuron: nn.Module,
        spike_compressor
    ):
        super().__init__()
        self.pool = pool
        self.proj = proj
        self.neuron = neuron
        self.spike_compressor = spike_compressor

    @staticmethod
    def conventional_forward(
        x_seq,
        pool_kernel_size,
        pool_stride,
        pool_padding,
        weight,
        bias,
        neuron,
        in_backward=False
    ):
        y_seq = avgpool1d_flatten_linear_forward(
            x_seq, pool_kernel_size, pool_stride, pool_padding, weight, bias
        )
        return neuron(y_seq)

    def forward(self, x_seq: torch.Tensor):
        return SNNCheckpointingBlockFunction.apply(
            self.conventional_forward,
            self.spike_compressor,
            x_seq,
            self.pool.kernel_size[0],
            self.pool.stride[0],
            self.pool.padding[0],
            self.proj.weight,
            self.proj.bias,
            self.neuron,
        )

With these GC segments, we can define the optimized SNN.

In [4]:
from modules.compress import get_spike_compressor

class GCSequentialCIFARNet(nn.Module):

    def __init__(
        self,
        channels: int,
        neuron_type: str,
        spike_compressor: str,
        num_classes=10,
        **kwargs
    ):
        """A Conv1d-based network for Sequential CIFAR-10/100 classification.

        Args:
            channels (int)
            neuron_type (str)
            spike_compressor (str)
            num_classes (int, optional): Defaults to 100.
            **kwargs: Additional arguments for `get_neuron(...)`. See 
                `src/models/neuron.py` for details.
        """
        super().__init__()

        conv = []
        for i in range(2):
            for j in range(3):
                if len(conv) == 0:
                    in_channels = 3
                else:
                    in_channels = channels

                if i == 0 and j == 0:
                    conv_block = [
                        Conv1dBNLIF(
                            proj=nn.Conv1d(
                                in_channels,
                                channels,
                                kernel_size=3,
                                padding=1,
                                bias=True
                            ),
                            bn=nn.BatchNorm1d(channels),
                            neuron=get_neuron(neuron_type, **kwargs),
                            spike_compressor=get_spike_compressor(
                                "NullSpikeCompressor"
                            ), # Input to the 1st layer is not binary. We shouldn't compress it.
                        )
                    ]
                elif i==1 and j == 0:
                    conv_block = [
                        AvgPool1dConv1dBNLIF(
                            pool=nn.AvgPool1d(2, 2),
                            proj=nn.Conv1d(
                                in_channels,
                                channels,
                                kernel_size=3,
                                padding=1,
                                bias=True
                            ),
                            bn=nn.BatchNorm1d(channels),
                            neuron=get_neuron(neuron_type, **kwargs),
                            spike_compressor=get_spike_compressor(
                                spike_compressor
                            ),
                        )
                    ]
                else:
                    conv_block = [
                        Conv1dBNLIF(
                            proj=nn.Conv1d(
                                in_channels,
                                channels,
                                kernel_size=3,
                                padding=1,
                                bias=True
                            ),
                            bn=nn.BatchNorm1d(channels),
                            neuron=get_neuron(neuron_type, **kwargs),
                            spike_compressor=get_spike_compressor(
                                spike_compressor
                            )
                        )
                    ]
                conv += conv_block

        self.conv = nn.Sequential(*conv)
        self.fc = AvgPool1dFlattenLinearLIF(
            pool=nn.AvgPool1d(2, 2),
            proj=nn.Linear(channels * 8, channels * 8 // 4),
            neuron=get_neuron(neuron_type, **kwargs),
            spike_compressor=get_spike_compressor(spike_compressor)
        )
        self.decode = nn.Linear(channels * 8 // 4, num_classes)

    def forward(self, x: torch.Tensor):
        x = x.permute(3, 0, 1, 2)
        # x.shape = [T, N, Cin, L]
        y = self.conv(x)
        y = self.fc(y)
        y = y.mean(dim=0)  # [N, C]
        y = self.decode(y)
        return y


hwlif_gc_net = GCSequentialCIFARNet(
    CHANNELS, 
    neuron_type="HandWrittenLIF", 
    spike_compressor="BitSpikeCompressor", 
    num_classes=10,
    decay_lambda=0.5
).to(DEVICE)

Train the network and record its peak memory usage.

In [5]:
optimizer = torch.optim.SGD(
    hwlif_gc_net.parameters(),
    lr=LR,
    momentum=MOMENTUM,
)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)


# reset peak memory recorder
torch.cuda.reset_peak_memory_stats(DEVICE)

for epoch in range(1): # only train 1 epoch
        train_results = train_step(
            hwlif_gc_net,
            train_loader,
            optimizer,
            lr_scheduler,
            DEVICE,
            epoch,
        )
        val_results = val_step(
            hwlif_gc_net,
            val_loader,
            DEVICE,
        )

        mem_stats = torch.cuda.memory_stats(DEVICE)
        peak_allocated = mem_stats["allocated_bytes.all.peak"] / (1024**2)
        peak_reserved = mem_stats["reserved_bytes.all.peak"] / (1024**2)

        print(
            f"Epoch {epoch + 1}: "
            f"train_loss={train_results['loss']}, "
            f"train_top1_acc={train_results['top1_acc']}, "
            f"val_loss={val_results['loss']}, "
            f"val_top1_acc={val_results['top1_acc']},\n\t"
            f"peak_allocated={peak_allocated} MB, "
            f"peak_reserved={peak_reserved} MB"
        )

Epoch 1: train_loss=2.060705163845649, train_top1_acc=25.94551282051282, val_loss=1.65302896232605, val_top1_acc=40.46,
	peak_allocated=481.74462890625 MB, peak_reserved=1846.0 MB


## Next Steps

1. **sGC**: Use `LayerWiseMemoryProfiler` in `src/utils/profiler.py` to profile the memory consumption of each layer. After finding the critical GC segment, split it into two sub-segments. Repeat this procedure until the peak memory cannot goes down. 
    * The resulting SNN is provided: `FGCSequentialCIFARNet` in `src/scifar/models.py`.

2. **pdsGC**: Use `LayerWiseFPCUDATimeProfiler` in `src/utils/profiler.py` to profile the inference time cost of each layer. Greedily disable the GC segment with the largest memory cost; if peak memory goes up, revert this change. Repeat this procedure until all segments are considered. 
    * The resulting SNN is provided: `PGCSequentialCIFARNet` in `src/scifar/models.py`.

3. Test the memory usage of these variants.